# Workspace CRUD Walkthrough

This notebook will:
1. Create a workspace
2. Fetch it by ID
3. Delete it
4. Verify it is gone

Prerequisites:
- PostgreSQL is running
- Migrations applied (e.g. `make migrate`)


In [1]:
from pathlib import Path
from uuid import uuid4
import sys

cwd = Path.cwd()
project_root = cwd.parent if cwd.name == 'playground' else cwd
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from dacke.domain.aggregates.workspace import Workspace
from dacke.domain.values.workspace import WorkspaceID
from dacke.infrastructure.config import AppSettings
from dacke.infrastructure.repositories.providers.postgres.repo_workspace import WorkspaceRepository


In [2]:
settings = AppSettings()
repo = WorkspaceRepository(connection_string=settings.database_url)

workspace_name = f'notebook-workspace-{uuid4().hex[:8]}'
workspace = Workspace.create(workspace_name)

await repo.create(workspace)
print('Created workspace:', workspace.identity, workspace.name)


2026-03-23 02:57:10,984 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-23 02:57:10,984 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-23 02:57:10,987 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-23 02:57:10,988 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-23 02:57:10,990 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-23 02:57:10,990 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-03-23 02:57:10,992 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 02:57:10,993 INFO sqlalchemy.engine.Engine INSERT INTO workspaces (name, id, created_at, updated_at) VALUES ($1::VARCHAR, $2::VARCHAR, $3::TIMESTAMP WITHOUT TIME ZONE, $4::TIMESTAMP WITHOUT TIME ZONE)
2026-03-23 02:57:10,994 INFO sqlalchemy.engine.Engine [generated in 0.00050s] ('notebook-workspace-dd32eb78', 'fdcaba95fb67599591eec71d9223f7a7', datetime.datetime(2026, 3, 23, 2, 57, 10, 876067), datetime.datetime(2026, 3, 23, 2, 57, 10, 876076))
2026-03-23 02:57:10,9

In [3]:
fetched = await repo.get_by_id(WorkspaceID.from_hex(str(workspace.identity)))
print('Fetched workspace:', fetched)


2026-03-23 02:57:15,184 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 02:57:15,186 INFO sqlalchemy.engine.Engine SELECT workspaces.name, workspaces.id, workspaces.created_at, workspaces.updated_at 
FROM workspaces 
WHERE workspaces.id = $1::VARCHAR
2026-03-23 02:57:15,187 INFO sqlalchemy.engine.Engine [generated in 0.00055s] ('fdcaba95fb67599591eec71d9223f7a7',)
2026-03-23 02:57:15,193 INFO sqlalchemy.engine.Engine SELECT collections.workspace_id AS collections_workspace_id, collections.name AS collections_name, collections.max_count_files AS collections_max_count_files, collections.max_file_size_kb AS collections_max_file_size_kb, collections.id AS collections_id, collections.created_at AS collections_created_at, collections.updated_at AS collections_updated_at, artifacts_1.collection_id AS artifacts_1_collection_id, artifacts_1.filename AS artifacts_1_filename, artifacts_1.source AS artifacts_1_source, artifacts_1.size_bytes AS artifacts_1_size_bytes, artifacts_1.mime_typ

In [4]:
if fetched is not None:
    await repo.delete(fetched)
    print('Deleted workspace:', fetched.identity)


2026-03-23 02:57:51,266 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 02:57:51,267 INFO sqlalchemy.engine.Engine SELECT workspaces.name, workspaces.id, workspaces.created_at, workspaces.updated_at 
FROM workspaces 
WHERE workspaces.id = $1::VARCHAR
2026-03-23 02:57:51,267 INFO sqlalchemy.engine.Engine [cached since 36.08s ago] ('fdcaba95fb67599591eec71d9223f7a7',)
2026-03-23 02:57:51,271 INFO sqlalchemy.engine.Engine SELECT collections.workspace_id AS collections_workspace_id, collections.name AS collections_name, collections.max_count_files AS collections_max_count_files, collections.max_file_size_kb AS collections_max_file_size_kb, collections.id AS collections_id, collections.created_at AS collections_created_at, collections.updated_at AS collections_updated_at, artifacts_1.collection_id AS artifacts_1_collection_id, artifacts_1.filename AS artifacts_1_filename, artifacts_1.source AS artifacts_1_source, artifacts_1.size_bytes AS artifacts_1_size_bytes, artifacts_1.mime_t

In [5]:
after_delete = await repo.get_by_id(WorkspaceID.from_hex(str(workspace.identity)))
print('After delete:', after_delete)

await repo._disconnect()


2026-03-23 02:57:59,420 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-23 02:57:59,421 INFO sqlalchemy.engine.Engine SELECT workspaces.name, workspaces.id, workspaces.created_at, workspaces.updated_at 
FROM workspaces 
WHERE workspaces.id = $1::VARCHAR
2026-03-23 02:57:59,422 INFO sqlalchemy.engine.Engine [cached since 44.23s ago] ('fdcaba95fb67599591eec71d9223f7a7',)
2026-03-23 02:57:59,424 INFO sqlalchemy.engine.Engine ROLLBACK
After delete: None
